Analyse der Erreichbarkeit mit dem ÖPNV (Busverkehr)

Zur Analyse der Verkehrserreichbarkeit des Busverkehrs in Schaidt wird die Erreichbarkeit der umliegenden Siedlungsflächen von einer Haltestelle des öffentlichen Personennahverkehrs (ÖPNV) aus untersucht. 

Der erste Python-Code dient dem Einlesen räumlicher Daten der Untersuchungsregion aus einer GeoJSON-Datei und deren Visualisierung auf einer interaktiven Karte. Zur Darstellung wird die Karte als HTML-Datei exportiert und anschließend in einem Webbrowser geöffnet. Hierzu werden zunächst die benötigten Python-Bibliotheken importiert, die sowohl die Verarbeitung geografischer Vektordaten als auch deren Darstellung im Webbrowser ermöglichen. Anschließend wird die GeoJSON-Datei eingelesen und als interaktive Karte gespeichert. Die zugrunde liegenden Rasterdaten beschränken sich auf Siedlungsflächen der betrachteten Gemeinden und Städte. Alle attributiven Informationen der Ausgangsdaten bleiben erhalten, um eine weiterführende Analyse zu ermöglichen.

In [1]:
import geopandas as gpd
import pandas as pd
import webbrowser

#Einlesen der GeoJSON-Datei mit den definierten Grids
filename = r"C:\Users\Sarah\Documents\Programme Studium\GeoInfo2\SVogel4.github.io\BA\Raster_BZA,Woerth.geojson"

gdf = gpd.read_file(filename)
g = gdf.explore()

file = "Untersuchungsregion.html"
g.save(file)

webbrowser.open(file)

True

Für die nachfolgenden Analysen werden repräsentative Punkte für jede Rasterzelle benötigt. Dazu werden die Zentroide der einzelnen Rasterzellen berechnet und als Punktgeometrien gespeichert. Hierzu erfolgt eine Transformation in das Koordinatenreferenzsystem EPSG:25832 (ETRS89/UTM-Zone 32N), da geometrische Operationen in einem metrischen Koordinatensystem durchgeführt werden sollten. Mit der Ausgabe der ersten fünf Zeilen des GeoDataFrames in einer Tabelle wird die Umwandlung in Punktgeometrien geprüft. 

In [2]:
points = gdf.copy()

# 1. CRS zuerst sicherstellen
points = points.to_crs(25832)

# 2. Zentroiden berechnen
points["geometry"] = points.geometry.centroid

# 3. Ergebnis prüfen
points.head()

,fid,rowid,featuretype_name,dataset_name,OBJECTID,id,x_sw,y_sw,x_mp,y_mp,f_staat,f_land,f_wasser,p_staat,p_land,p_wasser,Shape_Length,Shape_Area,ags,geometry
0,1,3538471,DE_Grid_ETRS89-LAEA_100m,de_grid_laea_100m,3538471,100mN28800E41870,4187000.0,2880000.0,4187050.0,2880050.0,10000.0,10000.0,0.0,100.0,100.0,0.0,400.0,10000.0,07334501,POINT (439241.002 5429860.233)
1,2,3538472,DE_Grid_ETRS89-LAEA_100m,de_grid_laea_100m,3538472,100mN28800E41871,4187100.0,2880000.0,4187150.0,2880050.0,10000.0,10000.0,0.0,100.0,100.0,0.0,400.0,10000.0,07334501,POINT (439340.934 5429861.624)
2,3,3538473,DE_Grid_ETRS89-LAEA_100m,de_grid_laea_100m,3538473,100mN28800E41872,4187200.0,2880000.0,4187250.0,2880050.0,10000.0,10000.0,0.0,100.0,100.0,0.0,400.0,10000.0,07334501,POINT (439440.867 5429863.016)
3,4,3538474,DE_Grid_ETRS89-LAEA_100m,de_grid_laea_100m,3538474,100mN28800E41873,4187300.0,2880000.0,4187350.0,2880050.0,10000.0,10000.0,0.0,100.0,100.0,0.0,400.0,10000.0,07334501,POINT (439540.799 5429864.407)
4,5,3538475,DE_Grid_ETRS89-LAEA_100m,de_grid_laea_100m,3538475,100mN28800E41874,4187400.0,2880000.0,4187450.0,2880050.0,10000.0,10000.0,0.0,100.0,100.0,0.0,400.0,10000.0,07334501,POINT (439640.731 5429865.799)


Die berechneten Zentroide werden anschließend in ein für Webkarten geeignetes Koordinatenreferenzsystem transformiert und anschließend ebenfalls interaktiv im Browser dargestellt. 

In [3]:
points = points.to_crs(4326)  # wichtig für Webkarten

# Interaktive Karte erstellen
m = points.explore()

# Karte als HTML-Datei speichern
html_file = "Punkte_Zentroiden.html"
m.save(html_file)

# HTML-Datei im Standardbrowser öffnen
webbrowser.open(html_file)


True

Der folgende Python-Code beschreibt die Erstellung eines integrierten Verkehrsnetzwerks unter Verwendung von Straßendaten aus OpenStreetMap (OSM) sowie von Fahrplandaten im GTFS-Format. Beide Datensätze liegen bereits in aufbereiteter Form vor und werden im ersten Schritt eingelesen. Die OSM-Daten enthalten Informationen zum Straßennetz und zur Verkehrsinfrastruktur. Die GTFS-Daten hingegen umfassen Fahrpläne und Linienverläufe des öffentlichen Nahverkehrs. In diesem Fall stammen die GTFS-Daten vom Verkehrsverbund Rhein-Neckar (VRN). Für die Erstellung eines integrierten Verkehrsnetzwerks in Schaidt werden die OSM-Daten und die GTFS-Datei kombiniert. Zur Einbindung wird die Funktion r5py.TransportNetwork aus der Python-Bibliothek r5py verwendet. Dies ist die Grundlage für spätere Erreichbarkeits- und Reisezeitanalysen.

In [4]:
from r5py import TransportNetwork

# Download der OSM-Daten 
osm_fp = r"C:\Users\Sarah\Documents\Programme Studium\GeoInfo2\SVogel4.github.io\BA\OSM_Daten.pbf"
transport_network = TransportNetwork(osm_fp)

# Download der GTFS-Daten
gtfs_fp = r"C:\Users\Sarah\Documents\Programme Studium\GeoInfo2\SVogel4.github.io\BA\vrn_gtfs_latest.zip"

# Erstellen des Verkehrsnetzwerks mit OSM- und GTFS-Daten
transport_network = TransportNetwork(
    # OSM data
    osm_fp, 
    
    # A list of GTFS file(s)
    [gtfs_fp]
)
print("Prepared the network")

Prepared the network


Im nächsten Schritt wird der Ausgangspunkt der Erreichbarkeitsanalyse festgelegt. Hierzu wird die Haltestelle „Schaidt, Rathaus“ geocodiert und ein Punktobjekt zur weiteren Analyse erstellt. Zur Geocodierung wird die Bibliothek OSMnx importiert, die auf Daten von OSM zurückgreift. Die Klasse Point aus der Bibliothek Shapely dient zur Erstellung von Punktgeometrien auf Basis von Koordinatenpaaren. Die ermittelten Koordinaten der Haltestelle werden in Lat und Lon angegeben, woraus ein Punktobjekt erzeugt wird. Dieses wird in einem GeoDataFrame gespeichert. Als geographisches Referenzsystem wird WGS84 zur Darstellung auf einer Webkarte verwendet. 

In [5]:
import osmnx as ox
from shapely.geometry import Point

# Geokodierung der Adresse "Schaidt, Rathaus" in Koordinaten
address = "Schaidt, Rathaus"
lat, lon = ox.geocode(address)
#lat = 49.055833
#lon = 8.086273




# Erstellen eines GeoDataFrames aus den Koordinaten
origin = gpd.GeoDataFrame({"geometry": [Point(lon, lat)], "name": "Schaidt, Rathaus", "id": [0]}, index=[0], crs="epsg:4326")

# Interaktive Karte erzeugen
m = origin.explore(
    max_zoom=13,
    color="red",
    marker_kwds={"radius": 12},
    )

# Karte als HTML-Datei speichern
html_file = "Vollmersweiler_Ort.html"
m.save(html_file)

# Karte im Standardbrowser öffnen
webbrowser.open(html_file)

True

Zur eindeutigen Identifizierung der Anfangs- und Endpunkte werden daraufhin fortlaufende Identifikationsnummern erzeugt. Hierzu wurde für jeden Datensatz eine numerische Kennung generiert. Mithilfe der IDs ist eine eindeutige Zuordnung der in den netzwerkbasierten Analysen berechneten Reisezeiten zu den jeweiligen Start- und Zielpunkten möglich. Insbesondere die Bibliothek r5py benötigt die IDs für die Erstellung von Reisezeitmatrizen.

In [6]:
origin["id"] = range(len(origin))
points["id"] = range(len(points))

Im nächsten Schritt wird die Reisezeitmatrix berechnet. Den Startort (Origin) stellt die Haltestelle in Schaidt dar, alle übrigen Punkte innerhalb des Untersuchungsgebietes sind die Zielorte (Destinations). Die Analyse wurde für den 23.06.2026 um 7:10 Uhr durchgeführt, da dies eine typische Pendlerzeit simuliert und zu dieser Zeit die meisten Schüler in die Schule fahren.  
Die ersten Zeilen der berechneten Reisezeitmatrix wurden angezeigt, um die Funktionsweise der Analyse zu prüfen. 

In [7]:
import datetime
from r5py import TravelTimeMatrix, TransportMode

# Berechnung der Reisezeitmatrix
travel_time_matrix = TravelTimeMatrix(
    transport_network,
    origins=origin,
    destinations=points,
    max_time=datetime.timedelta(minutes=120), # Maximale Reisezeit: 120 Minuten
    departure=datetime.datetime(2026, 6, 23, 7, 10), #Abfahrtszeit: 23. Juni 2026, 07:10 Uhr
    transport_modes=[
        TransportMode.TRANSIT, # Allgemeiner öffentlicher Verkehr
        TransportMode.RAIL,  # Regional- und Fernzüge
        TransportMode.SUBWAY,  # U-Bahn
        TransportMode.TRAM # Straßenbahn
    ],  
    max_time_walking=datetime.timedelta(minutes=7),        
    
)

travel_time_matrix.head()

,from_id,to_id,travel_time
0,0,0,0.0
1,0,1,99.0
2,0,2,42.0
3,0,3,42.0
4,0,4,42.0


In dem nächsten Codeabschnitt wird die berechnete Reisezeitmatrix mit dem Grid verknüpft, um die Reisezeiten für die einzelnen Zellen zu ermitteln. Dadurch ist eine räumliche Visualisierung der Erreichbarkeit auf einer Karte möglich. Zunächst wird hierzu ein neuer Index erzeugt, falls dieser zuvor möglicherweise nicht fortlaufend oder eindeutig war. Anschließend wird der aktuelle Index in eine neue Spalte „id” kopiert, um eine Zuordnung der Reisezeiten aus der Matrix mit den entsprechenden Polygonflächen zu gewährleisten. Anschließend werden die beiden Tabellen zusammengeführt, wozu die Spalten „id” und „to_id” verwendet werden.. 
Um das Ergebnis der Verknüpfung zu kontrollieren, werden die ersten fünf Zeilen des Ergebnisses angezeigt. 

In [9]:
gdf = gdf.reset_index(drop=True)
gdf["id"] = gdf.index

# Verknüpfung der Geodaten mit der Reisezeitmatrix
join = gdf.merge(
    travel_time_matrix,
    left_on="id",
    right_on="to_id",
    how="left"
)

join.head()


,fid,rowid,featuretype_name,dataset_name,OBJECTID,id,x_sw,y_sw,x_mp,y_mp,...,p_staat,p_land,p_wasser,Shape_Length,Shape_Area,ags,geometry,from_id,to_id,travel_time
0,1,3538471,DE_Grid_ETRS89-LAEA_100m,de_grid_laea_100m,3538471,0,4187000.0,2880000.0,4187050.0,2880050.0,...,100.0,100.0,0.0,400.0,10000.0,07334501,"POLYGON ((439191.707 5429809.548, 439190.364 5...",0,0,0.0
1,2,3538472,DE_Grid_ETRS89-LAEA_100m,de_grid_laea_100m,3538472,1,4187100.0,2880000.0,4187150.0,2880050.0,...,100.0,100.0,0.0,400.0,10000.0,07334501,"POLYGON ((439291.639 5429810.939, 439290.297 5...",0,1,99.0
2,3,3538473,DE_Grid_ETRS89-LAEA_100m,de_grid_laea_100m,3538473,2,4187200.0,2880000.0,4187250.0,2880050.0,...,100.0,100.0,0.0,400.0,10000.0,07334501,"POLYGON ((439391.572 5429812.331, 439390.229 5...",0,2,42.0
3,4,3538474,DE_Grid_ETRS89-LAEA_100m,de_grid_laea_100m,3538474,3,4187300.0,2880000.0,4187350.0,2880050.0,...,100.0,100.0,0.0,400.0,10000.0,07334501,"POLYGON ((439491.504 5429813.723, 439490.162 5...",0,3,42.0
4,5,3538475,DE_Grid_ETRS89-LAEA_100m,de_grid_laea_100m,3538475,4,4187400.0,2880000.0,4187450.0,2880050.0,...,100.0,100.0,0.0,400.0,10000.0,07334501,"POLYGON ((439591.436 5429815.114, 439590.094 5...",0,4,42.0


Ausgabe der gültigen Werte zur Analyse der Ergebnisse

In [10]:
valid = join["travel_time"].dropna()

print("Gültige Werte:", valid.shape[0])
print("Fehlende Werte:", join["travel_time"].isna().sum())

print("Durchschnitt (Min):", valid.mean())
print("Median (Min):", valid.median())

Gültige Werte: 955
Fehlende Werte: 651
Durchschnitt (Min): 45.556020942408374
Median (Min): 49.0


Mithilfe des letzten Codeabschnitts wird eine interaktive Karte erstellt, die die Reisezeit zu verschiedenen Standorten in der Untersuchungsregion visualisiert. Dies erfolgt anhand einer Choropletenkarte, bei der die einzelnen Zellen basierend auf ihrer Reisezeit zum Startpunkt farblich kodiert werden. Die Farben reichen hierbei von grün für kurze Reisezeiten bis rot für längere Reisezeiten. Die Zeitintervalle sind auf 0–5 Minuten, 5–10 Minuten, 10–20 Minuten, 20–30 Minuten und über 30 Minuten festgelegt. Dies wird durch die Funktion pd.cut() aus der Bibliothek Pandas ermöglicht. Zur benutzerfreundlichen Darstellung wird der Kartenmittelpunkt aus allen vorhandenen Rasterzellen berechnet und deren Mittelpunkt festgelegt. Die Farben zeigen intuitiv, welche Orte gut und welche weniger gut erreichbar sind. Beim Überfahren einer Fläche mit dem Mauszeiger wird die zugehörige Reisezeit in Minuten angezeigt. Dazu werden die klassifizierten Polygonflächen als GeoJSON-Layer in die Karte integriert. Da Folium keine automatische Funktion dazu bereitstellt, wird die Legende manuell mit HTML und CSS erstellt. Die fertige Karte wurde als HTML-Datei gespeichert und zur weiteren Analyse im Webbrowser bereitgestellt.

In [11]:
import folium

# 2. Reisezeit sauber in Minuten
join["travel_min"] = join["travel_time"]


# 3. Klassenbildung (WICHTIG für Legende)
join["travel_class"] = pd.cut(
    join["travel_min"],
    bins=[0, 5, 10, 20, 30, float("inf")],
    labels=["0-5", "5-10", "10-20", "20-30", ">30"],
    include_lowest=True
)

# 4. CRS für webbasierten Kartendienst (Folium)
join_4326 = join.to_crs(4326)


# Gestaltung der Karte 

# 5. Kartenmittelpunkt
center_geom = join_4326.geometry.union_all().centroid
center = [center_geom.y, center_geom.x]

# 6. Farben
colors = {
    "0-5": "#006837",
    "5-10": "#5eaa7f",
    "10-20": "#ffdd57",
    "20-30": "#fd8d3c",
    ">30": "#bd0026"
}


# 7. Style Funktion
def style_function(feature):
    val = feature["properties"]["travel_class"]
    return {
        "fillColor": colors.get(val, "#d9d9d9"),
        "color": "black",
        "weight": 0.2,
        "fillOpacity": 0.7
    }


# 8. Karte
m = folium.Map(
    location=center,
    zoom_start=13,
    tiles="CartoDB positron"
)


# 9. GeoJson Layer
folium.GeoJson(
    join_4326,
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=["travel_min"],
        aliases=["Reisezeit (Min):"]
    )
).add_to(m)


# 10. Legende (manuell)
legend_html = """
<div style="
position: fixed;
bottom: 50px;
left: 50px;
width: 230px;
background: white;
border: 2px solid grey;
z-index: 9999;
font-size: 14px;
padding: 10px;
">

<b>ÖPNV-Reisezeit</b><br>
<b>Abfahrt: 07:10 Uhr</b><br><br>

<div><span style="background:#006837;width:18px;height:18px;display:inline-block;"></span> 0–5 Min</div>
<div><span style="background:#5eaa7f;width:18px;height:18px;display:inline-block;"></span> 5–10 Min</div>
<div><span style="background:#ffdd57;width:18px;height:18px;display:inline-block;"></span> 10–20 Min</div>
<div><span style="background:#fd8d3c;width:18px;height:18px;display:inline-block;"></span> 20–30 Min</div>
<div><span style="background:#bd0026;width:18px;height:18px;display:inline-block;"></span> >30 Min</div>
</div>
"""

m.get_root().html.add_child(folium.Element(legend_html))

# 11. Speichern
map_path = "karte.html"
m.save(map_path)
webbrowser.open(map_path)

True

In [13]:
#7. Style Funktion
def style_function(feature):
    val = feature["properties"]["travel_class"]
    return {
      "fillColor": colors.get(val, "#d9d9d9"),
       "color": "black",
       "weight": 0.2,
       "fillOpacity": 0.7
   }


# GeoPackage speichern


join_4326.to_file(
    "reisezeitenVollmersweiler7_10.gpkg",
    layer="reisezeiten",
    driver="GPKG"
)
print("GeoPackage wurde erfolgreich gespeichert.")

GeoPackage wurde erfolgreich gespeichert.
